# Practical 6: POS Tagging using Hidden Markov Models (HMM)

**Concept:** **Part-of-Speech (POS) tagging** assigns a grammatical tag (noun, verb, adjective, ...) to every word in a sentence. This is a classic **sequence labelling** problem, ideally suited for a **Hidden Markov Model (HMM)**.

An HMM models the sentence as:
- **Hidden states** = the POS tags we want to predict.
- **Observations** = the words we actually see.

It is fully described by three parameters (matrices):

1. **Initial distribution π** — probability that a sentence starts with tag *t*: `P(t_1)`
2. **Transition matrix A** — probability of moving from tag *i* to tag *j*: `P(t_j | t_i)`
3. **Emission matrix B** — probability of seeing word *w* given tag *t*: `P(w | t)`

Given a sentence, we find the **most probable tag sequence** using the **Viterbi algorithm**, a dynamic-programming decoding method.

> **Important:** In this practical we implement the HMM **from scratch** — we estimate the π, A, B matrices by counting, and we code the Viterbi algorithm ourselves (no pre-built tagger is used for the decoding).

## Import Required Libraries
We only need `math` for logarithms, `pandas` for pretty matrix display and `nltk` for a comparison at the very end.

In [15]:
%pip install pandas


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/tejas/.local/share/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math

import pandas as pd

import nltk
from nltk import pos_tag, word_tokenize

## Download NLTK Resources
Only needed for the final NLTK comparison step.

In [3]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package punkt to /home/tejas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tejas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/tejas/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

## Create the Tagged Corpus (Training Data)
We create a small corpus where every sentence is already tagged with a POS label for each word. This is our **training data** — we will count from it to build the HMM matrices.

In [4]:
# Each sentence is a list of (word, POS_tag) tuples
tagged_corpus = [
    [("the", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("the", "DET"), ("dog", "NOUN"), ("barks", "VERB"), ("loudly", "ADV")],
    [("a", "DET"), ("beautiful", "ADJ"), ("flower", "NOUN"), ("blooms", "VERB")],
    [("john", "NOUN"), ("loves", "VERB"), ("mary", "NOUN")],
    [("mary", "NOUN"), ("runs", "VERB"), ("fast", "ADV")],
    [("the", "DET"), ("young", "ADJ"), ("boy", "NOUN"), ("plays", "VERB")],
    [("she", "PRON"), ("reads", "VERB"), ("a", "DET"), ("book", "NOUN")],
    [("he", "PRON"), ("eats", "VERB"), ("rice", "NOUN")],
    [("the", "DET"), ("happy", "ADJ"), ("girl", "NOUN"), ("smiles", "VERB")],
    [("cats", "NOUN"), ("eat", "VERB"), ("fish", "NOUN")]
]

print("Tagged Corpus")
print("-" * 50)
for i, sentence in enumerate(tagged_corpus, start=1):
    print(f"Sentence {i}: {sentence}")

Tagged Corpus
--------------------------------------------------
Sentence 1: [('the', 'DET'), ('cat', 'NOUN'), ('sleeps', 'VERB')]
Sentence 2: [('the', 'DET'), ('dog', 'NOUN'), ('barks', 'VERB'), ('loudly', 'ADV')]
Sentence 3: [('a', 'DET'), ('beautiful', 'ADJ'), ('flower', 'NOUN'), ('blooms', 'VERB')]
Sentence 4: [('john', 'NOUN'), ('loves', 'VERB'), ('mary', 'NOUN')]
Sentence 5: [('mary', 'NOUN'), ('runs', 'VERB'), ('fast', 'ADV')]
Sentence 6: [('the', 'DET'), ('young', 'ADJ'), ('boy', 'NOUN'), ('plays', 'VERB')]
Sentence 7: [('she', 'PRON'), ('reads', 'VERB'), ('a', 'DET'), ('book', 'NOUN')]
Sentence 8: [('he', 'PRON'), ('eats', 'VERB'), ('rice', 'NOUN')]
Sentence 9: [('the', 'DET'), ('happy', 'ADJ'), ('girl', 'NOUN'), ('smiles', 'VERB')]
Sentence 10: [('cats', 'NOUN'), ('eat', 'VERB'), ('fish', 'NOUN')]


## Extract Tag Set & Vocabulary
The **tag set** are the hidden states of our HMM; the **vocabulary** are the possible observations.

In [5]:
# All unique tags (hidden states)
tags = sorted(set(tag for sentence in tagged_corpus for _, tag in sentence))

# All unique words (observations)
vocabulary = sorted(set(word for sentence in tagged_corpus for word, _ in sentence))

print("Tag Set (Hidden States)")
print("-" * 50)
print(tags)

print("\nVocabulary (Observations)")
print("-" * 50)
print(vocabulary)

Tag Set (Hidden States)
--------------------------------------------------
['ADJ', 'ADV', 'DET', 'NOUN', 'PRON', 'VERB']

Vocabulary (Observations)
--------------------------------------------------
['a', 'barks', 'beautiful', 'blooms', 'book', 'boy', 'cat', 'cats', 'dog', 'eat', 'eats', 'fast', 'fish', 'flower', 'girl', 'happy', 'he', 'john', 'loudly', 'loves', 'mary', 'plays', 'reads', 'rice', 'runs', 'she', 'sleeps', 'smiles', 'the', 'young']


## Step 1: Count Everything
We count three things directly from the training corpus:
1. `start_counts` — how often each tag begins a sentence
2. `transition_counts` — how often tag *i* is immediately followed by tag *j*
3. `emission_counts` — how often tag *t* is paired with word *w*

In [6]:
# 1. How often each tag starts a sentence
start_counts = {tag: 0 for tag in tags}

# 2. How often tag_i is followed by tag_j
transition_counts = {}

# 3. How often a tag emits a word
emission_counts = {}

for sentence in tagged_corpus:
    # First word's tag
    start_counts[sentence[0][1]] += 1

    # Tag transitions within the sentence
    for i in range(len(sentence) - 1):
        prev_tag = sentence[i][1]
        next_tag = sentence[i + 1][1]
        key = (prev_tag, next_tag)
        transition_counts[key] = transition_counts.get(key, 0) + 1

    # Word emission for each tagged word
    for word, tag in sentence:
        key = (tag, word)
        emission_counts[key] = emission_counts.get(key, 0) + 1

print("Counts Collected")
print("-" * 50)
print("Start counts:", start_counts)
print("Transition counts:", transition_counts)
print("Emission counts:", emission_counts)

Counts Collected
--------------------------------------------------
Start counts: {'ADJ': 0, 'ADV': 0, 'DET': 5, 'NOUN': 3, 'PRON': 2, 'VERB': 0}
Transition counts: {('DET', 'NOUN'): 3, ('NOUN', 'VERB'): 8, ('VERB', 'ADV'): 2, ('DET', 'ADJ'): 3, ('ADJ', 'NOUN'): 3, ('VERB', 'NOUN'): 3, ('PRON', 'VERB'): 2, ('VERB', 'DET'): 1}
Emission counts: {('DET', 'the'): 4, ('NOUN', 'cat'): 1, ('VERB', 'sleeps'): 1, ('NOUN', 'dog'): 1, ('VERB', 'barks'): 1, ('ADV', 'loudly'): 1, ('DET', 'a'): 2, ('ADJ', 'beautiful'): 1, ('NOUN', 'flower'): 1, ('VERB', 'blooms'): 1, ('NOUN', 'john'): 1, ('VERB', 'loves'): 1, ('NOUN', 'mary'): 2, ('VERB', 'runs'): 1, ('ADV', 'fast'): 1, ('ADJ', 'young'): 1, ('NOUN', 'boy'): 1, ('VERB', 'plays'): 1, ('PRON', 'she'): 1, ('VERB', 'reads'): 1, ('NOUN', 'book'): 1, ('PRON', 'he'): 1, ('VERB', 'eats'): 1, ('NOUN', 'rice'): 1, ('ADJ', 'happy'): 1, ('NOUN', 'girl'): 1, ('VERB', 'smiles'): 1, ('NOUN', 'cats'): 1, ('VERB', 'eat'): 1, ('NOUN', 'fish'): 1}


## Step 2: Initial Distribution π
The probability that a sentence **starts** with tag *t* is estimated as:

$$
\pi(t) = \frac{count(start=t) + 1}{N_{sentences} + |tags|}
$$

We add **1 to every count** (add-one / Laplace smoothing) so no probability is ever zero.

In [7]:
# Number of sentences in the corpus
num_sentences = len(tagged_corpus)

# Smoothed initial distribution
pi = {}
for tag in tags:
    pi[tag] = (start_counts[tag] + 1) / (num_sentences + len(tags))

print("Initial Distribution π (smoothed)")
print("-" * 50)
for tag, prob in pi.items():
    print(f"P(start = {tag:<5}) = {prob:.4f}")

Initial Distribution π (smoothed)
--------------------------------------------------
P(start = ADJ  ) = 0.0625
P(start = ADV  ) = 0.0625
P(start = DET  ) = 0.3750
P(start = NOUN ) = 0.2500
P(start = PRON ) = 0.1875
P(start = VERB ) = 0.0625


## Step 3: Transition Matrix A
The probability of moving from tag *i* to tag *j* is:

$$
A[i][j] = \frac{count(t_i \to t_j) + 1}{count(t_i) + |tags|}
$$

where `count(t_i)` is the total number of times tag *i* appears as a *previous* tag.

In [8]:
# Total times each tag appears as the previous tag
tag_prev_totals = {tag: 0 for tag in tags}
for (prev, _), count in transition_counts.items():
    tag_prev_totals[prev] += count

# Smoothed transition matrix
A = {}
for prev in tags:
    for nxt in tags:
        count = transition_counts.get((prev, nxt), 0)
        A[(prev, nxt)] = (count + 1) / (tag_prev_totals[prev] + len(tags))

# Display as a DataFrame for clarity
transition_df = pd.DataFrame(
    [[A[(prev, nxt)] for nxt in tags] for prev in tags],
    index=tags,
    columns=tags
)

print("Transition Matrix A (rows = previous tag, columns = next tag)")
print("-" * 50)
print(transition_df.round(4))

Transition Matrix A (rows = previous tag, columns = next tag)
--------------------------------------------------
         ADJ     ADV     DET    NOUN    PRON    VERB
ADJ   0.1111  0.1111  0.1111  0.4444  0.1111  0.1111
ADV   0.1667  0.1667  0.1667  0.1667  0.1667  0.1667
DET   0.3333  0.0833  0.0833  0.3333  0.0833  0.0833
NOUN  0.0714  0.0714  0.0714  0.0714  0.0714  0.6429
PRON  0.1250  0.1250  0.1250  0.1250  0.1250  0.3750
VERB  0.0833  0.2500  0.1667  0.3333  0.0833  0.0833


## Step 4: Emission Matrix B
The probability of seeing word *w* given tag *t* is:

$$
B[t][w] = \frac{count(t, w) + 1}{count(t) + |vocabulary|}
$$

Unknown words simply receive the smoothing constant, which lets the model make a (weak) guess even for unseen words.

In [9]:
# Total times each tag emits a word
tag_emit_totals = {tag: 0 for tag in tags}
for (tag, _), count in emission_counts.items():
    tag_emit_totals[tag] += count

# Smoothed emission matrix
B = {}
for tag in tags:
    for word in vocabulary:
        count = emission_counts.get((tag, word), 0)
        B[(tag, word)] = (count + 1) / (tag_emit_totals[tag] + len(vocabulary))

# Display a few rows as a DataFrame
emission_df = pd.DataFrame(
    [[B[(tag, word)] for word in vocabulary] for tag in tags],
    index=tags,
    columns=vocabulary
)

print("Emission Matrix B (rows = tag, columns = word)")
print("-" * 50)
print(emission_df.round(4))

Emission Matrix B (rows = tag, columns = word)
--------------------------------------------------
           a   barks  beautiful  blooms    book     boy     cat    cats  \
ADJ   0.0303  0.0303     0.0606  0.0303  0.0303  0.0303  0.0303  0.0303   
ADV   0.0312  0.0312     0.0312  0.0312  0.0312  0.0312  0.0312  0.0312   
DET   0.0833  0.0278     0.0278  0.0278  0.0278  0.0278  0.0278  0.0278   
NOUN  0.0238  0.0238     0.0238  0.0238  0.0476  0.0476  0.0476  0.0476   
PRON  0.0312  0.0312     0.0312  0.0312  0.0312  0.0312  0.0312  0.0312   
VERB  0.0250  0.0500     0.0250  0.0500  0.0250  0.0250  0.0250  0.0250   

         dog     eat  ...    mary   plays   reads    rice    runs     she  \
ADJ   0.0303  0.0303  ...  0.0303  0.0303  0.0303  0.0303  0.0303  0.0303   
ADV   0.0312  0.0312  ...  0.0312  0.0312  0.0312  0.0312  0.0312  0.0312   
DET   0.0278  0.0278  ...  0.0278  0.0278  0.0278  0.0278  0.0278  0.0278   
NOUN  0.0476  0.0238  ...  0.0714  0.0238  0.0238  0.0476  0.0238  0

## Step 5: The Viterbi Algorithm (from scratch)

Given an observation sequence (words), Viterbi finds the **most likely hidden state sequence** (tags). We work in **log space** to avoid underflow when multiplying many small probabilities.

Algorithm:
1. **Initialization:** for each tag *s*, score = `log(π[s]) + log(emission)` of the first word.
2. **Recursion:** for each next word, for each tag *s*, keep the *best* previous tag and accumulate `log(transition) + log(emission)`.
3. **Backtracking:** trace the best path backwards to recover the full tag sequence.

In [10]:
def emission_prob(tag, word):
    """Return P(word | tag) with smoothing.
    Unknown words get the smoothing constant of the tag."""
    return B.get((tag, word), 1.0 / (tag_emit_totals[tag] + len(vocabulary)))


def viterbi(words):
    """Find the most probable POS tag sequence for a list of words.
    Returns (best_tag_sequence, log_probability)."""
    n = len(words)
    T = len(tags)

    # Viterbi table and back-pointer table
    viterbi_table = [[-math.inf] * T for _ in range(n)]
    backpointer = [[None] * T for _ in range(n)]

    # --- Initialization: first word ---
    for s, tag in enumerate(tags):
        viterbi_table[0][s] = math.log(pi[tag]) + math.log(emission_prob(tag, words[0]))

    # --- Recursion: remaining words ---
    for t in range(1, n):
        for s, tag in enumerate(tags):
            # Find the previous tag that maximizes the score
            best_prev = max(
                range(T),
                key=lambda s_prev: viterbi_table[t - 1][s_prev] + math.log(A[(tags[s_prev], tag)])
            )
            viterbi_table[t][s] = viterbi_table[t - 1][best_prev] + math.log(A[(tags[best_prev], tag)]) + math.log(emission_prob(tag, words[t]))
            backpointer[t][s] = best_prev

    # --- Termination: best final tag ---
    best_last = max(range(T), key=lambda s: viterbi_table[n - 1][s])
    best_log_prob = viterbi_table[n - 1][best_last]

    # --- Backtracking ---
    best_path = [best_last]
    for t in range(n - 1, 0, -1):
        best_path.insert(0, backpointer[t][best_path[0]])

    # Convert state indices back to tag names
    best_tags = [tags[idx] for idx in best_path]
    return best_tags, best_log_prob


print("Viterbi function defined successfully!")

Viterbi function defined successfully!


## Step 6: Tag the Training Sentences
We decode the same sentences used for training. The HMM should recover their tags perfectly (or nearly so).

In [11]:
print("HMM Predictions on Training Sentences")
print("-" * 50)

for i, sentence in enumerate(tagged_corpus, start=1):
    words = [word for word, _ in sentence]
    true_tags = [tag for _, tag in sentence]
    pred_tags, log_prob = viterbi(words)

    match = "OK" if pred_tags == true_tags else "X"
    print(f"Sentence {i}: {' '.join(words)}")
    print(f"  True: {true_tags}")
    print(f"  Pred: {pred_tags}  [{match}]")
    print(f"  Log-probability: {log_prob:.4f}")
    print()

HMM Predictions on Training Sentences
--------------------------------------------------
Sentence 1: the cat sleeps
  True: ['DET', 'NOUN', 'VERB']
  Pred: ['DET', 'NOUN', 'VERB']  [OK]
  Log-probability: -10.5356

Sentence 2: the dog barks loudly
  True: ['DET', 'NOUN', 'VERB', 'ADV']
  Pred: ['DET', 'NOUN', 'VERB', 'ADV']  [OK]
  Log-probability: -14.6945

Sentence 3: a beautiful flower blooms
  True: ['DET', 'ADJ', 'NOUN', 'VERB']
  Pred: ['DET', 'ADJ', 'NOUN', 'VERB']  [OK]
  Log-probability: -14.6607

Sentence 4: john loves mary
  True: ['NOUN', 'VERB', 'NOUN']
  Pred: ['NOUN', 'VERB', 'NOUN']  [OK]
  Log-probability: -11.6061

Sentence 5: mary runs fast
  True: ['NOUN', 'VERB', 'ADV']
  Pred: ['NOUN', 'VERB', 'ADV']  [OK]
  Log-probability: -11.6218

Sentence 6: the young boy plays
  True: ['DET', 'ADJ', 'NOUN', 'VERB']
  Pred: ['DET', 'ADJ', 'NOUN', 'VERB']  [OK]
  Log-probability: -14.1499

Sentence 7: she reads a book
  True: ['PRON', 'VERB', 'DET', 'NOUN']
  Pred: ['PRON', 'V

## Step 7: Tag a New Sentence
Now we test on a sentence **not** in the training corpus. Note how unknown words (like *"sings"* and *"little"*) are handled with smoothing.

In [12]:
new_sentence = "the little cat sings"

# Tokenize manually (or with NLTK)
new_words = new_sentence.lower().split()

# Decode with our HMM
pred_tags, log_prob = viterbi(new_words)

print("Tagging a New Sentence")
print("-" * 50)

for word, tag in zip(new_words, pred_tags):
    print(f"{word:<10} -> {tag}")

print(f"\nLog-probability of best path: {log_prob:.4f}")

Tagging a New Sentence
--------------------------------------------------
the        -> DET
little     -> ADJ
cat        -> NOUN
sings      -> VERB

Log-probability of best path: -15.5362


## Step 8: Compare with NLTK's Tagger
For reference, we compare our from-scratch HMM output with NLTK's pre-trained POS tagger. NLTK uses a larger tag set (Penn Treebank), so the tag *names* differ, but the grammatical role should agree.

In [13]:
# NLTK's own POS tagger
nltk_tags = pos_tag(new_words)

print("Comparison: Our HMM vs NLTK")
print("-" * 50)

print(f"{'Word':<10}{'Our HMM':<12}{'NLTK (Penn)'}")
for i, word in enumerate(new_words):
    print(f"{word:<10}{pred_tags[i]:<12}{nltk_tags[i][1]}")

Comparison: Our HMM vs NLTK
--------------------------------------------------
Word      Our HMM     NLTK (Penn)
the       DET         DT
little    ADJ         JJ
cat       NOUN        NN
sings     VERB        NNS


## Step 9: Accuracy Evaluation
We evaluate the HMM on a small **held-out** tagged set of sentences and compute the token-level accuracy.

In [14]:
# Small held-out test set (not used during training)
test_corpus = [
    [("the", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("a", "DET"), ("happy", "ADJ"), ("boy", "NOUN"), ("runs", "VERB")],
    [("he", "PRON"), ("reads", "VERB"), ("a", "DET"), ("book", "NOUN")]
]

# Collect all true tags and all predicted tags
true_all = []
pred_all = []

print("Evaluation on Held-out Sentences")
print("-" * 50)

for i, sentence in enumerate(test_corpus, start=1):
    words = [word for word, _ in sentence]
    true_tags = [tag for _, tag in sentence]
    pred_tags, _ = viterbi(words)

    true_all.extend(true_tags)
    pred_all.extend(pred_tags)

    match = "OK" if pred_tags == true_tags else "X"
    print(f"Test {i}: {' '.join(words)}")
    print(f"  True: {true_tags}")
    print(f"  Pred: {pred_tags}  [{match}]")
    print()

# Token-level accuracy
correct = sum(1 for t, p in zip(true_all, pred_all) if t == p)
accuracy = correct / len(true_all)

print("-" * 50)
print(f"Token-level Accuracy: {correct}/{len(true_all)} = {accuracy:.2%}")

Evaluation on Held-out Sentences
--------------------------------------------------
Test 1: the dog sleeps
  True: ['DET', 'NOUN', 'VERB']
  Pred: ['DET', 'NOUN', 'VERB']  [OK]

Test 2: a happy boy runs
  True: ['DET', 'ADJ', 'NOUN', 'VERB']
  Pred: ['DET', 'ADJ', 'NOUN', 'VERB']  [OK]

Test 3: he reads a book
  True: ['PRON', 'VERB', 'DET', 'NOUN']
  Pred: ['PRON', 'VERB', 'DET', 'NOUN']  [OK]

--------------------------------------------------
Token-level Accuracy: 11/11 = 100.00%


## Summary

In this practical we implemented a **POS tagger based on an HMM entirely from scratch**:
1. Built the three HMM parameters — **π** (initial), **A** (transition) and **B** (emission) — by counting from a tagged corpus with add-one smoothing.
2. Implemented the **Viterbi algorithm** by hand using log-space dynamic programming.
3. Tagged both training and new unseen sentences.
4. Compared against NLTK's pre-trained tagger.
5. Measured token-level accuracy on a held-out set.

This demonstrates the core idea behind statistical sequence labelling, and how a small HMM can already learn meaningful grammar patterns from very little data.